# People Identification in the Wild — FAISS Edition

This notebook is identical in objective and dataset to `people_identification_in_the_wild.ipynb`  
**but replaces the manual cosine-distance loop with FAISS** for faster nearest-neighbour matching.

Key differences from the original notebook:
- `faiss-cpu` is installed alongside standard dependencies
- A FAISS `IndexFlatIP` index is built from L2-normalised gallery embeddings
- All identity lookups (sections 6.5, 7, 8) use a single `index.search()` call instead of a Python loop over the gallery
- A timing comparison cell shows the speedup vs the original loop

Everything else (detector, threshold, evaluation logic, CSV export) is identical.

## 0) Install and import dependencies

In [ ]:
# Uncomment to install if needed:
# !pip install deepface opencv-python matplotlib pandas tqdm tf-keras faiss-cpu

import os
import zipfile
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss                        # FAISS — replaces manual cosine loop

from tqdm.auto import tqdm
from deepface import DeepFace

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

print("Imports OK — FAISS version:", faiss.__version__)

## 1) Locate and prepare dataset

In [ ]:
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "open_data_set"

zip_files = sorted(PROJECT_ROOT.glob("*.zip"))
if zip_files:
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    for z in zip_files:
        print(f"Extracting {z.name} -> {DATASET_ROOT}")
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(DATASET_ROOT)
else:
    print("No zip found — assuming dataset already extracted.")

expected_folders = ["photos_all", "photos_all_faces", "portraits", "trio_cam", "trio_gp"]
for folder in expected_folders:
    fp = DATASET_ROOT / folder
    imgs = list(fp.glob("*.jpg")) + list(fp.glob("*.JPG")) + list(fp.glob("*.png"))
    print(f"{folder:16s} -> exists={fp.exists()} | images={len(imgs)}")

## 2) Build metadata table

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

rows = []

# ---- Gallery: photos_all_faces (pre-cropped face images, one per identity per photo)
gallery_dir = DATASET_ROOT / "photos_all_faces"
if gallery_dir.exists():
    for img_path in sorted(gallery_dir.iterdir()):
        if img_path.suffix not in IMAGE_EXTS:
            continue
        name = img_path.stem.upper()
        # Filename convention: "A_001.jpg" or "A001.jpg" — first character is identity
        identity = name[0]
        rows.append({"path": str(img_path), "split": "gallery", "identity": identity})

# ---- Query: portraits (close-up face images used to build query signatures)
query_dir = DATASET_ROOT / "portraits"
if query_dir.exists():
    for img_path in sorted(query_dir.iterdir()):
        if img_path.suffix not in IMAGE_EXTS:
            continue
        name = img_path.stem.upper()
        identity = name[0]
        rows.append({"path": str(img_path), "split": "query", "identity": identity})

# ---- Scene pool: trio_cam and trio_gp are the drone/group scene images
for scene_folder in ["trio_cam", "trio_gp"]:
    d = DATASET_ROOT / scene_folder
    if d.exists():
        for img_path in sorted(d.iterdir()):
            if img_path.suffix not in IMAGE_EXTS:
                continue
            rows.append({"path": str(img_path), "split": "scene_pool", "identity": None})

meta_df = pd.DataFrame(rows)
print("Total files indexed:", len(meta_df))
print(meta_df["split"].value_counts())
print("\nDistinct identities in gallery:", sorted(meta_df[meta_df["split"] == "gallery"]["identity"].unique()))
meta_df.head()

## 3) Choose query image and scene image

In [ ]:
# Change TARGET_ID to whichever identity you want to search for.
TARGET_ID = "A"

query_candidates = meta_df[(meta_df["split"] == "query") & (meta_df["identity"] == TARGET_ID)]
if len(query_candidates) == 0:
    query_candidates = meta_df[(meta_df["split"] == "gallery") & (meta_df["identity"] == TARGET_ID)]
if len(query_candidates) == 0:
    raise RuntimeError(f"No query/gallery images found for identity '{TARGET_ID}'.")

query_path = query_candidates.iloc[0]["path"]
print("Query image:", query_path)

# Pick a scene that should contain the target identity.
scene_token = TARGET_ID.lower()
matching_scenes = meta_df[
    (meta_df["split"] == "scene_pool") &
    (meta_df["path"].str.contains(r'(?i)' + scene_token, regex=True))
]
if len(matching_scenes) == 0:
    matching_scenes = meta_df[meta_df["split"] == "scene_pool"]

scene_path = matching_scenes.iloc[0]["path"]
print("Scene image:", scene_path)

# Sanity check — display both side by side.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(cv2.imread(query_path), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Query — Identity {TARGET_ID}")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(cv2.imread(scene_path), cv2.COLOR_BGR2RGB))
axes[1].set_title("Scene image (drone view)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4) Build gallery embeddings and FAISS index

This is the key difference from the original notebook.

### Why FAISS?

In the original notebook, finding the nearest gallery identity for a face requires:
```
for each gallery embedding:
    compute cosine_distance(scene_emb, gallery_emb)
```
This is **O(N)** per query face — fine for a small gallery but slow at scale.

FAISS (`IndexFlatIP` + L2 normalisation) reduces this to a **single matrix operation** that runs on optimised BLAS/SIMD instructions (or GPU if `faiss-gpu` is installed).

Mathematically: cosine similarity = dot product of **L2-normalised** vectors.  
So we:
1. L2-normalise every gallery embedding → store in `IndexFlatIP`
2. L2-normalise the query vector → call `index.search(query, k=1)`
3. FAISS returns `(inner_product, gallery_index)` → cosine_distance = 1 − inner_product

In [ ]:
MODEL_NAME       = "ArcFace"   # Recognition backbone — must stay the same across gallery & query
DISTANCE_METRIC  = "cosine"    # We replicate cosine via inner product on L2-normalised vectors

MAX_PER_ID = 30  # Cap gallery images per identity for faster development runs


def get_embedding(image_path: str, model_name: str = MODEL_NAME,
                  detector_backend: str = "skip") -> np.ndarray | None:
    """
    Extract a single embedding from a pre-cropped face image.

    detector_backend='skip' tells DeepFace not to run a detector —
    the image is already a cropped face, so detection is unnecessary.
    """
    try:
        reps = DeepFace.represent(
            img_path=image_path,
            model_name=model_name,
            detector_backend=detector_backend,
            enforce_detection=False,
        )
        if not reps:
            return None
        return np.array(reps[0]["embedding"], dtype=np.float32)
    except Exception as e:
        print(f"Embedding failed for {image_path}: {e}")
        return None


def l2_normalize(v: np.ndarray) -> np.ndarray:
    """L2-normalise a vector so that dot(a, b) == cosine_similarity(a, b)."""
    norm = np.linalg.norm(v)
    return v / (norm + 1e-8)


# Build gallery DataFrame (deduplicated to MAX_PER_ID images per identity).
gallery_meta = meta_df[meta_df["split"] == "gallery"].copy()
gallery_meta = (
    gallery_meta.groupby("identity", group_keys=False)
    .head(MAX_PER_ID)
    .reset_index(drop=True)
)

raw_embeddings = []
identities     = []

for _, row in tqdm(gallery_meta.iterrows(), total=len(gallery_meta), desc="Gallery embeddings"):
    emb = get_embedding(row["path"])
    if emb is not None:
        raw_embeddings.append(l2_normalize(emb))
        identities.append(row["identity"])

gallery_matrix = np.stack(raw_embeddings).astype(np.float32)  # shape: (N, D)
gallery_ids    = np.array(identities)                          # shape: (N,)

print(f"Gallery size: {len(gallery_ids)} embeddings, dimension: {gallery_matrix.shape[1]}")
print("Unique identities:", sorted(set(gallery_ids)))

In [ ]:
# ----------------------------
# BUILD THE FAISS INDEX
# ----------------------------
DIM = gallery_matrix.shape[1]   # Embedding dimensionality (e.g. 512 for ArcFace)

# IndexFlatIP: exact search on inner product (= cosine similarity for L2-normed vectors).
# For very large galleries (>100k) you'd swap this for IndexIVFFlat or IndexHNSWFlat
# for approximate-nearest-neighbour (ANN) search at a small accuracy cost.
faiss_index = faiss.IndexFlatIP(DIM)

# Add all gallery vectors at once — this is a simple flat index, no training needed.
faiss_index.add(gallery_matrix)

print(f"FAISS index built — total vectors: {faiss_index.ntotal}")
print(f"Index type: IndexFlatIP (exact cosine search via inner product)")


def faiss_nearest(query_emb: np.ndarray, k: int = 1):
    """
    Return top-k gallery matches for a single query embedding.

    Returns:
        List of dicts: [{"identity": str, "cosine_distance": float}, ...]
    """
    q = l2_normalize(query_emb).reshape(1, -1).astype(np.float32)
    # FAISS returns (similarities, indices) for IndexFlatIP.
    sims, idxs = faiss_index.search(q, k)
    results = []
    for sim, idx in zip(sims[0], idxs[0]):
        if idx == -1:           # FAISS returns -1 when fewer vectors exist than k
            continue
        results.append({
            "identity": gallery_ids[idx],
            "cosine_distance": float(1.0 - sim),   # convert similarity → distance
        })
    return results


print("\nSelf-test — nearest neighbour of gallery_matrix[0] should be gallery_ids[0]:")
test = faiss_nearest(gallery_matrix[0], k=3)
for t in test:
    print(f"  -> {t}")

## 5) Extract query signature `qs`

In [ ]:
query_embedding = get_embedding(query_path)

if query_embedding is None:
    raise RuntimeError("Could not extract query embedding — try another query image.")

print(f"Query embedding extracted | length: {len(query_embedding)}")

# FAISS nearest-neighbour check on query itself — should return TARGET_ID as best match.
top3 = faiss_nearest(query_embedding, k=3)
print(f"\nFAISS top-3 gallery matches for query (identity={TARGET_ID}):")
for r in top3:
    print(f"  identity={r['identity']}  cosine_distance={r['cosine_distance']:.4f}")

## 6) FAISS vs loop speed comparison

Before running detection, let's benchmark how much faster FAISS is compared to the original Python loop approach on the gallery we built.

In [ ]:
import timeit

# --- Baseline: original Python loop (cosine distance per gallery embedding)
def loop_search(query_emb: np.ndarray) -> tuple[str, float]:
    """Original loop-based nearest-neighbour search (O(N) Python)."""
    q = l2_normalize(query_emb)
    best_dist, best_id = float("inf"), None
    for i, g_emb in enumerate(gallery_matrix):
        # inner product of two L2-normalised vectors = cosine similarity
        dist = float(1.0 - np.dot(q, g_emb))
        if dist < best_dist:
            best_dist, best_id = dist, gallery_ids[i]
    return best_id, best_dist


# --- FAISS search (single BLAS matrix multiply)
def faiss_search(query_emb: np.ndarray) -> tuple[str, float]:
    top1 = faiss_nearest(query_emb, k=1)
    return top1[0]["identity"], top1[0]["cosine_distance"]


REPEATS = 50   # Run each method 50 times and average

t_loop  = timeit.timeit(lambda: loop_search(query_embedding),  number=REPEATS) / REPEATS
t_faiss = timeit.timeit(lambda: faiss_search(query_embedding), number=REPEATS) / REPEATS

print(f"Gallery size:      {len(gallery_ids)} vectors  |  dimension: {DIM}")
print(f"Loop  avg time:    {t_loop*1000:.3f} ms")
print(f"FAISS avg time:    {t_faiss*1000:.3f} ms")
print(f"Speedup:           {t_loop/t_faiss:.1f}x  (FAISS is faster)")

# Verify both return the same identity.
id_loop,  d_loop  = loop_search(query_embedding)
id_faiss, d_faiss = faiss_search(query_embedding)
print(f"\nLoop  result:  identity={id_loop},  distance={d_loop:.4f}")
print(f"FAISS result:  identity={id_faiss}, distance={d_faiss:.4f}")
print("Results match:", id_loop == id_faiss)

## 6.4) Fixed detector setup: OpenCV @ scale 2.0

Same detector strategy as the original notebook — `opencv` backend at 2× upscale to detect small distant faces.

In [ ]:
FIXED_DETECTOR_BACKEND = "opencv"
FIXED_DETECTOR_SCALE   = 2.0

scene_bgr = cv2.imread(scene_path)
if scene_bgr is None:
    raise FileNotFoundError(f"Could not read scene: {scene_path}")
scene_h, scene_w = scene_bgr.shape[:2]


def detect_scene_boxes(scene_img: np.ndarray,
                        backend: str = FIXED_DETECTOR_BACKEND,
                        scale: float = FIXED_DETECTOR_SCALE) -> list[dict]:
    """
    Run DeepFace detection on an upscaled version of the scene, then
    map bounding boxes back to original resolution.

    Returns list of {"x", "y", "w", "h"} dicts (original image coords).
    """
    h0, w0 = scene_img.shape[:2]
    scaled = cv2.resize(scene_img, None, fx=scale, fy=scale,
                        interpolation=cv2.INTER_CUBIC)
    try:
        raw_faces = DeepFace.extract_faces(
            img_path=scaled,
            detector_backend=backend,
            enforce_detection=True,
            align=True,
        )
    except Exception:
        raw_faces = []

    boxes = []
    for item in raw_faces:
        area = item["facial_area"]
        x = max(0, min(int(area["x"] / scale), w0 - 1))
        y = max(0, min(int(area["y"] / scale), h0 - 1))
        w = max(1, min(int(area["w"] / scale), w0 - x))
        h = max(1, min(int(area["h"] / scale), h0 - y))
        if w >= 10 and h >= 10:
            boxes.append({"x": x, "y": y, "w": w, "h": h})
    return boxes


fixed_boxes = detect_scene_boxes(scene_bgr)
print(f"Detector: {FIXED_DETECTOR_BACKEND} @ scale {FIXED_DETECTOR_SCALE}  |  faces found: {len(fixed_boxes)}")

if len(fixed_boxes) == 0:
    raise RuntimeError("No faces detected. Try another scene image or adjust FIXED_DETECTOR_SCALE.")

# Visualise bounding boxes on the scene image.
vis = cv2.cvtColor(scene_bgr.copy(), cv2.COLOR_BGR2RGB)
for b in fixed_boxes:
    cv2.rectangle(vis, (b["x"], b["y"]), (b["x"]+b["w"], b["y"]+b["h"]), (0, 255, 0), 2)

plt.figure(figsize=(11, 6))
plt.imshow(vis)
plt.title(f"Detected faces | {FIXED_DETECTOR_BACKEND} @ scale {FIXED_DETECTOR_SCALE}")
plt.axis("off")
plt.show()

## 6.5) Show detected faces with predicted IDs (FAISS lookup)

In [ ]:
face_id_rows = []

for idx, b in enumerate(fixed_boxes):
    crop_bgr = scene_bgr[b["y"]:b["y"]+b["h"], b["x"]:b["x"]+b["w"]]
    if crop_bgr.size == 0:
        continue
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    try:
        reps = DeepFace.represent(
            img_path=crop_rgb,
            model_name=MODEL_NAME,
            detector_backend="skip",
            enforce_detection=False,
        )
        if not reps:
            continue
        emb = np.array(reps[0]["embedding"], dtype=np.float32)
    except Exception:
        continue

    # FAISS nearest-neighbour lookup — returns top-1 gallery match.
    top1 = faiss_nearest(emb, k=1)[0]
    face_id_rows.append({
        "face_idx":  idx,
        "pred_id":   top1["identity"],
        "cos_dist":  round(top1["cosine_distance"], 4),
        **b,
    })

face_id_df = pd.DataFrame(face_id_rows)
print("Detected faces with FAISS IDs:")
display(face_id_df)

# Annotate scene image with ID labels.
annotated = scene_bgr.copy()
for _, r in face_id_df.iterrows():
    x, y, w, h = int(r.x), int(r.y), int(r.w), int(r.h)
    cv2.rectangle(annotated, (x, y), (x+w, y+h), (0, 220, 0), 2)
    label = f"{r.pred_id}:{r.cos_dist:.2f}"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(annotated, (x, max(0, y-22)), (x+tw+6, y), (0, 220, 0), -1)
    cv2.putText(annotated, label, (x+3, y-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1, cv2.LINE_AA)

out_all = str(PROJECT_ROOT / "faiss_output_all_detected_faces_with_ids.jpg")
cv2.imwrite(out_all, annotated)
print(f"Saved: {out_all}")

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("All detected faces with FAISS-predicted IDs")
plt.axis("off")
plt.show()

## 7) Match query to scene — find TARGET_ID with FAISS

**Pipeline (matches project objectives):**
1. For each detected face → extract embedding
2. FAISS `index.search()` → nearest gallery identity + cosine distance
3. Also compute distance from the specific *query* embedding to each face
4. The face with the smallest query distance (below threshold) is the match

In [ ]:
MATCH_THRESHOLD = 0.20   # Cosine distance threshold — lower means stricter

results = []

for idx, b in enumerate(fixed_boxes):
    crop_bgr = scene_bgr[b["y"]:b["y"]+b["h"], b["x"]:b["x"]+b["w"]]
    if crop_bgr.size == 0:
        continue
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    try:
        reps = DeepFace.represent(
            img_path=crop_rgb,
            model_name=MODEL_NAME,
            detector_backend="skip",
            enforce_detection=False,
        )
        if not reps:
            continue
        scene_emb = np.array(reps[0]["embedding"], dtype=np.float32)
    except Exception:
        continue

    # Distance from the specific query image to this face.
    q = l2_normalize(query_embedding)
    s = l2_normalize(scene_emb)
    d_query = float(1.0 - np.dot(q, s))    # cosine distance (scalar — fast)

    # FAISS gallery lookup — who does this face look like?
    top1 = faiss_nearest(scene_emb, k=1)[0]

    results.append({
        "face_index":       idx,
        "query_distance":   round(d_query, 4),
        "predicted_id":     top1["identity"],
        "gallery_distance": round(top1["cosine_distance"], 4),
        "x": b["x"], "y": b["y"], "w": b["w"], "h": b["h"],
    })

results_df = pd.DataFrame(results).sort_values("query_distance").reset_index(drop=True)
print("Detected faces with valid embeddings:", len(results_df))
display(results_df.head(10))

if len(results_df) == 0:
    raise RuntimeError("No embeddings extracted from scene faces.")

best = results_df.iloc[0]
is_match = best["query_distance"] <= MATCH_THRESHOLD

print(f"\nBest candidate:  face_index={best.face_index}  "
      f"predicted_id={best.predicted_id}  query_dist={best.query_distance}")
print(f"Match decision (dist ≤ {MATCH_THRESHOLD}): {is_match}")

# Annotate the scene with a green bounding box + label.
out = scene_bgr.copy()
x, y, w, h = int(best.x), int(best.y), int(best.w), int(best.h)
cv2.rectangle(out, (x, y), (x+w, y+h), (0, 255, 0), 3)
label = f"ID={best.predicted_id} | qDist={best.query_distance:.3f}"
if not is_match:
    label = "NO MATCH | " + label
(tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
cv2.rectangle(out, (x, max(0, y-28)), (x+tw+8, y), (0, 255, 0), -1)
cv2.putText(out, label, (x+4, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2, cv2.LINE_AA)

out_path = str(PROJECT_ROOT / "faiss_output_query_localization.jpg")
cv2.imwrite(out_path, out)
print(f"\nSaved annotated output: {out_path}")

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.title("Final Output: localised and identified query face (FAISS)")
plt.axis("off")
plt.show()

## 8) Full per-scene evaluation with ground truth, TP/FP, precision, recall (FAISS)

Identical evaluation logic to the original notebook — but all gallery lookups use FAISS.  
Results are exported to `faiss_batch_evaluation_results.csv`.

In [ ]:
MATCH_THRESHOLD = 0.20   # Keep consistent with section 7

scene_paths = meta_df[meta_df["split"] == "scene_pool"]["path"].tolist()


def token_to_expected_ids(token: str) -> set:
    """
    Map a scene filename token to the set of expected person IDs.
    e.g. "EF" -> {"E", "F"},  "IJK" -> {"I", "J", "K"}
    """
    return set(list(token.upper()))


scene_rows = []
face_rows  = []

t_batch_start = time.time()

for sp in tqdm(scene_paths, desc="Batch eval (FAISS)"):
    token = Path(sp).stem.split("_")[0].upper()
    expected_ids = token_to_expected_ids(token)

    scene_img = cv2.imread(sp)
    if scene_img is None:
        scene_rows.append({
            "scene_path": sp, "scene_token": token,
            "expected_ids": str(sorted(expected_ids)),
            "num_detected": 0, "all_predicted_ids": "",
            "true_positives": 0, "false_positives": 0,
            "precision": np.nan, "recall": np.nan, "error": "could_not_read",
        })
        continue

    # Detect faces with fixed detector.
    boxes = detect_scene_boxes(scene_img)

    predicted_ids_in_scene = []

    for box in boxes:
        crop_bgr = scene_img[box["y"]:box["y"]+box["h"], box["x"]:box["x"]+box["w"]]
        if crop_bgr.size == 0:
            continue
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

        try:
            reps = DeepFace.represent(
                img_path=crop_rgb,
                model_name=MODEL_NAME,
                detector_backend="skip",
                enforce_detection=False,
            )
            if not reps:
                continue
            emb = np.array(reps[0]["embedding"], dtype=np.float32)
        except Exception:
            continue

        # ---- FAISS lookup (replaces the old gallery loop) ----
        top1 = faiss_nearest(emb, k=1)[0]
        pred_id  = top1["identity"]
        pred_dist = top1["cosine_distance"]

        accepted = pred_dist <= MATCH_THRESHOLD
        if accepted:
            predicted_ids_in_scene.append(pred_id)

        face_rows.append({
            "scene_path":              sp,
            "scene_token":             token,
            "expected_ids":            str(sorted(expected_ids)),
            "predicted_id":            pred_id,
            "nearest_gallery_distance": pred_dist,
            "accepted":                accepted,
            "is_true_positive":        pred_id in expected_ids,
            "x": box["x"], "y": box["y"], "w": box["w"], "h": box["h"],
        })

    predicted_set = set(predicted_ids_in_scene)
    tp = len(predicted_set & expected_ids)
    fp = len(predicted_set - expected_ids)
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall    = tp / len(expected_ids) if len(expected_ids) > 0 else np.nan

    scene_rows.append({
        "scene_path":       sp,
        "scene_token":      token,
        "expected_ids":     str(sorted(expected_ids)),
        "num_detected":     len(boxes),
        "all_predicted_ids": str(sorted(predicted_set)),
        "true_positives":   tp,
        "false_positives":  fp,
        "precision":        round(precision, 3) if not np.isnan(precision) else np.nan,
        "recall":           round(recall,    3) if not np.isnan(recall)    else np.nan,
        "error":            None,
    })

t_batch_total = time.time() - t_batch_start

scene_eval_df = pd.DataFrame(scene_rows)
face_eval_df  = pd.DataFrame(face_rows)

csv_path = str(PROJECT_ROOT / "faiss_batch_evaluation_results.csv")
scene_eval_df.to_csv(csv_path, index=False)
print(f"Results saved to: {csv_path}")
print(f"Total batch time: {t_batch_total:.1f}s  ({t_batch_total/max(len(scene_paths),1):.2f}s/scene)")

print(f"\nTotal scenes evaluated:  {len(scene_eval_df)}")
print(f"Scenes with ≥1 TP:       {(scene_eval_df['true_positives'] > 0).sum()}")
print(f"Mean precision:          {scene_eval_df['precision'].mean():.3f}")
print(f"Mean recall:             {scene_eval_df['recall'].mean():.3f}")

print("\nPer-scene results (sorted by recall ↓):")
display(scene_eval_df.sort_values("recall", ascending=False).head(30))

print("\nPer group-token summary:")
group_summary = (
    scene_eval_df.groupby("scene_token")
    .agg(
        scenes=("scene_path", "count"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_tp=("true_positives", "mean"),
        mean_fp=("false_positives", "mean"),
    )
    .round(3)
    .reset_index()
)
display(group_summary)

## 10) Objective checklist

| Objective | Section | Notes |
|---|---|---|
| Detect all faces in drone scene | 6.4 | OpenCV backend @ 2× scale |
| Crop detected faces | 6.4 | Per-box crop from original resolution |
| Extract face signatures (s1…sn) | 4 + 6.5 | ArcFace embeddings, L2-normalised for FAISS |
| Extract query signature (qs) | 5 | Same ArcFace model |
| Compute distances d1…dn | 7 | FAISS `IndexFlatIP` — cosine via inner product |
| Select minimum distance face | 7 | `results_df.iloc[0]` (sorted by query_distance) |
| Output bounding box + ID | 7 | Green box + label on scene image, saved to disk |
| Batch evaluation (TP/FP/precision/recall) | 8 | CSV exported to `faiss_batch_evaluation_results.csv` |
| FAISS vs loop speed comparison | 6 | Section 6 timing cell |